**1 - Install Dependencies**

In [3]:
%pip install osmnx networkx geopy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


**2 - Imports**

In [4]:
import osmnx as ox
import networkx as nx
import pandas as pd
from geopy.distance import geodesic
from geopy.geocoders import Nominatim

**3 - Inputs From Previous Agents**

In [ ]:
# Agent 1 Output
#agent1_output = {
#   "location":"Delhi",
#   "disaster_type":"Flood",
#   "rainfall":120,
#   "temperature":28,
#   "water_level":8.5,
#   "social_reports":340,
#   "emergency_calls":120
#}

# Agent 2 Output
#agent2_output = {
#    "severity":"Extreme",
#    "affected_population":50000,
#    "damage_score":92
#}

# Agent 3 Output
#agent3_output = {
#    "rescue_teams":49,
#    "ambulances":9,
#    "medical_units":4,
#    "food_packets":50000
#}

#city = agent1_output["location"] + ", India"

#damage = agent2_output

#resources = agent3_output

#disaster_type = agent1_output["disaster_type"]

In [6]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(
    user_agent="disaster_management_system"
)

location = geolocator.geocode(city)

user_lat = location.latitude
user_lon = location.longitude

print("Location:", city)
print("Latitude:", user_lat)
print("Longitude:", user_lon)

Location: Delhi, India
Latitude: 28.6138954
Longitude: 77.2090057


**4 - Shelter Database**

In [7]:
print("Discovering facilities from OpenStreetMap...")

schools = ox.features_from_place(
    city,
    tags={"amenity": "school"}
)

hospitals = ox.features_from_place(
    city,
    tags={"amenity": "hospital"}
)

community_centres = ox.features_from_place(
    city,
    tags={"amenity": "community_centre"}
)

print("Schools:", len(schools))
print("Hospitals:", len(hospitals))
print("Community Centres:", len(community_centres))

facilities = pd.concat([
    schools,
    hospitals,
    community_centres
])

facilities = facilities.reset_index()

print(
    "Total facilities:",
    len(facilities)
)

Discovering facilities from OpenStreetMap...
Schools: 131
Hospitals: 41
Community Centres: 24
Total facilities: 196


In [8]:
from geopy.distance import geodesic

FACILITIES = []

for _, row in facilities.iterrows():

    try:

        name = row.get("name")

        if pd.isna(name):
            continue

        amenity = row.get("amenity")

        geom = row.geometry

        if geom.geom_type == "Point":

            lat = geom.y
            lon = geom.x

        else:

            center = geom.centroid

            lat = center.y
            lon = center.x

        FACILITIES.append({

            "name": str(name),

            "type": str(amenity),

            "lat": float(lat),

            "lon": float(lon)

        })

    except:
        continue

# Remove duplicate facility names

seen = set()

unique_facilities = []

for facility in FACILITIES:

    if facility["name"] not in seen:

        seen.add(facility["name"])

        unique_facilities.append(facility)

FACILITIES = unique_facilities

print(
    f"Facilities after deduplication: {len(FACILITIES)}"
)

# Compute straight-line distance

for facility in FACILITIES:

    facility["straight_line_distance"] = geodesic(

        (
            user_lat,
            user_lon
        ),

        (
            facility["lat"],
            facility["lon"]
        )

    ).km

FACILITIES.sort(

    key=lambda x:
    x["straight_line_distance"]

)

MAX_CANDIDATES = 50

FACILITIES = FACILITIES[:MAX_CANDIDATES]

print(
    f"Keeping nearest {len(FACILITIES)} facilities"
)

print("\nTop 10 closest facilities:\n")

for facility in FACILITIES[:10]:

    print(

        f"{facility['name']} "

        f"({facility['type']}) "

        f"- {facility['straight_line_distance']:.2f} km"

    )

Facilities after deduplication: 163
Keeping nearest 50 facilities

Top 10 closest facilities:

Press Club of India (community_centre) - 0.58 km
Young Women's Christian Association (community_centre) - 1.39 km
Guru Harkrishan Polyclinic Bangla Sahib (hospital) - 1.43 km
Rashtrapati Bhavan Cultural Center (community_centre) - 1.46 km
YMCA Cultural Centre Building (community_centre) - 1.49 km
Dr. Ram Manohar Lohia Hospital (hospital) - 1.56 km
Lady Irwin Senior Secondary School (school) - 1.61 km
Convent of Jesus and Mary (school) - 1.69 km
General Williams Hospital (hospital) - 1.69 km
Lycée Français de Delhi (school) - 1.72 km


**5 - Route Planning Agent**

In [9]:
class RoutePlanningAgent:

    def plan_route(
        self,
        city,
        user_lat,
        user_lon,
        damage,
        resources
    ):

        print("Downloading road network...")

        G = ox.graph_from_place(
            city,
            network_type="drive"
        )

        origin = ox.distance.nearest_nodes(
            G,
            user_lon,
            user_lat
        )

        routes = []

        facility_scores = {

            "hospital": 6,

            "community_centre": 4,

            "school": 3

        }

        for facility in FACILITIES:

            try:

                destination = ox.distance.nearest_nodes(

                    G,

                    facility["lon"],

                    facility["lat"]

                )

                route = nx.shortest_path(

                    G,

                    origin,

                    destination,

                    weight="length"

                )

                route_length = nx.shortest_path_length(

                    G,

                    origin,

                    destination,

                    weight="length"

                )

                distance_km = round(

                    route_length / 1000,

                    2

                )

                eta_min = round(

                    route_length / 500

                )

                coordinates = []

                for node in route:

                    coordinates.append({

                        "lat":
                        G.nodes[node]["y"],

                        "lon":
                        G.nodes[node]["x"]

                    })

                facility_score = facility_scores.get(

                    facility["type"],

                    1

                )

                distance_score = max(

                    0,

                    10 - distance_km

                )

                total_score = (

                    0.7 * distance_score

                    +

                    0.3 * facility_score

                )

                routes.append({

                    "facility":
                    facility["name"],

                    "facility_type":
                    facility["type"],

                    "distance_km":
                    distance_km,

                    "eta_min":
                    eta_min,

                    "score":
                    round(total_score, 2),

                    "coordinates":
                    coordinates

                })

            except Exception as e:

                print(

                    f"Error with {facility['name']}:",

                    e

                )

                continue

        if len(routes) == 0:

            raise Exception(
                "No valid routes found."
            )

        routes.sort(

            key=lambda x: x["score"],

            reverse=True

        )

        best_route = routes[0]

        alternative_routes = []

        for route in routes[1:6]:

            alternative_routes.append({

                "facility":
                route["facility"],

                "facility_type":
                route["facility_type"],

                "distance_km":
                route["distance_km"],

                "eta_min":
                route["eta_min"],

                "score":
                route["score"]

            })

        result = {

            "severity":
            damage["severity"],

            "affected_population":
            damage["affected_population"],

            "allocated_resources":
            resources,

            "recommended_route": {

                "facility":
                best_route["facility"],

                "facility_type":
                best_route["facility_type"],

                "distance_km":
                best_route["distance_km"],

                "eta_min":
                best_route["eta_min"],

                "score":
                best_route["score"]

            },

            "alternative_routes":
            alternative_routes,

            "recommended_route_coordinates":
            best_route["coordinates"]

        }

        return result

**6 - Run Agent 4**

In [10]:
agent = RoutePlanningAgent()

route_output = agent.plan_route(

    city=city,

    user_lat=user_lat,

    user_lon=user_lon,

    damage=damage,

    resources=resources

)

route_output

{'severity': 'Extreme',
 'affected_population': 50000,
 'allocated_resources': {'rescue_teams': 49,
  'ambulances': 9,
  'medical_units': 4,
  'food_packets': 50000},
 'recommended_route': {'facility': 'General Williams Hospital',
  'facility_type': 'hospital',
  'distance_km': np.float64(2.26),
  'eta_min': 5,
  'score': np.float64(7.22)},
 'alternative_routes': [{'facility': 'Guru Harkrishan Polyclinic Bangla Sahib',
   'facility_type': 'hospital',
   'distance_km': np.float64(2.46),
   'eta_min': 5,
   'score': np.float64(7.08)},
  {'facility': 'Press Club of India',
   'facility_type': 'community_centre',
   'distance_km': np.float64(1.71),
   'eta_min': 3,
   'score': np.float64(7.0)},
  {'facility': 'Arvind Medicare',
   'facility_type': 'hospital',
   'distance_km': np.float64(2.63),
   'eta_min': 5,
   'score': np.float64(6.96)},
  {'facility': "Young Women's Christian Association",
   'facility_type': 'community_centre',
   'distance_km': np.float64(1.9),
   'eta_min': 4,
   '

**7 - Cleaner Display**

In [11]:
print("\n===== EVACUATION PLAN =====\n")

print(
    "Recommended Facility:",
    route_output["recommended_route"]["facility"]
)

print(
    "Facility Type:",
    route_output["recommended_route"]["facility_type"]
)

print(
    "Distance:",
    route_output["recommended_route"]["distance_km"],
    "km"
)

print(
    "ETA:",
    route_output["recommended_route"]["eta_min"],
    "minutes"
)

print(
    "Score:",
    route_output["recommended_route"]["score"]
)

print("\nAlternative Routes:\n")

for i, route in enumerate(
    route_output["alternative_routes"],
    start=1
):

    print(

        f"{i}. "

        f"{route['facility']} "

        f"({route['facility_type']}) "

        f"| {route['distance_km']} km "

        f"| ETA {route['eta_min']} min "

        f"| Score {route['score']}"

    )


===== EVACUATION PLAN =====

Recommended Facility: General Williams Hospital
Facility Type: hospital
Distance: 2.26 km
ETA: 5 minutes
Score: 7.22

Alternative Routes:

1. Guru Harkrishan Polyclinic Bangla Sahib (hospital) | 2.46 km | ETA 5 min | Score 7.08
2. Press Club of India (community_centre) | 1.71 km | ETA 3 min | Score 7.0
3. Arvind Medicare (hospital) | 2.63 km | ETA 5 min | Score 6.96
4. Young Women's Christian Association (community_centre) | 1.9 km | ETA 4 min | Score 6.87
5. Dr. Ram Manohar Lohia Hospital (hospital) | 2.86 km | ETA 6 min | Score 6.8


**8 - Peoplesense integration**

In [12]:
import requests
import math

# ======================================
# Fetch PeopleSense Occupancy Data
# ======================================

url = "https://w8bdwhaps0.execute-api.us-west-2.amazonaws.com/v1/occupancy?filter=ALL"

headers = {
    "x-api-key": "KplI1ustKlYvY14YfAm02XTdwMAEfmE46IHqbpQ5"
}

try:

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    occupancy_data = response.json()["data"]

except Exception as e:

    print("Could not fetch PeopleSense data.")

    print(e)

    occupancy_data = []


# ======================================
# Haversine Distance
# ======================================

def haversine(lat1, lon1, lat2, lon2):

    R = 6371

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)

    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (

        math.sin(dlat / 2) ** 2

        +

        math.cos(lat1)

        *

        math.cos(lat2)

        *

        math.sin(dlon / 2) ** 2

    )

    c = 2 * math.atan2(

        math.sqrt(a),

        math.sqrt(1 - a)

    )

    return R * c


# ======================================
# Find Nearby Transit Locations
# ======================================

nearby_transit = []

for place in occupancy_data:

    lat = place.get("Latitude")

    lon = place.get("Longitude")

    if lat in [None, 0] or lon in [None, 0]:

        continue

    distance = haversine(

        user_lat,

        user_lon,

        float(lat),

        float(lon)

    )

    if distance <= 5:

        occupancy = place.get("Occupancy")

        max_occ = place.get("MaxOccupancy")

        percent = None

        if occupancy is not None:

            if max_occ and max_occ > 0:

                percent = round(
                    occupancy / max_occ * 100,
                    1
                )

            else:

                percent = occupancy

        nearby_transit.append({

            "name":

            place.get("GroupID")
            or place.get("PlaceID")
            or "Unknown",

            "distance":

            round(distance,2),

            "occupancy_percent":

            percent

        })

nearby_transit.sort(

    key=lambda x: x["distance"]

)

route_output["nearby_transit"] = nearby_transit[:5]

In [22]:
import pandas as pd

best = route_output["recommended_route"]

print("\n" + "="*70)
print("                 EVACUATION PLAN")
print("="*70)

# ==========================================================
# Recommended Facility
# ==========================================================

print("\n🏥 RECOMMENDED FACILITY")
print("-"*70)

recommended_df = pd.DataFrame([{
    "Facility": best["facility"],
    "Type": best["facility_type"].replace("_"," ").title(),
    "Distance (km)": best["distance_km"],
    "ETA (min)": best["eta_min"],
    "Score": best["score"]
}])

display(recommended_df)

# ==========================================================
# Alternative Facilities
# ==========================================================

print("\n🏥 ALTERNATIVE FACILITIES")
print("-"*70)

alt_df = pd.DataFrame(route_output["alternative_routes"])

alt_df.rename(columns={
    "facility":"Facility",
    "facility_type":"Type",
    "distance_km":"Distance (km)",
    "eta_min":"ETA (min)",
    "score":"Score"
}, inplace=True)

alt_df["Type"] = (
    alt_df["Type"]
    .str.replace("_"," ")
    .str.title()
)

display(alt_df)

# ==========================================================
# Nearby Transit Occupancy
# ==========================================================

print("\n🚇 NEARBY TRANSIT CONGESTION (PeopleSense)")
print("-"*70)

rows = []

for place in route_output["nearby_transit"]:

    occ = place["occupancy_percent"]

    if occ is None:
        continue

    if occ <= 25:
        status = "🟢 Low"

    elif occ <= 50:
        status = "🟡 Moderate"

    elif occ <= 75:
        status = "🟠 Busy"

    else:
        status = "🔴 Crowded"

    rows.append({

        "Transit Hub": place["name"],

        "Distance (km)": place["distance"],

        "Occupancy (%)": occ,

        "Status": status

    })

if len(rows):

    transit_df = pd.DataFrame(rows)

    display(transit_df)

else:

    print("No nearby transit occupancy available.")

# ==========================================================
# Travel Advisory
# ==========================================================

print("\n⚠ TRAVEL ADVISORY")
print("-"*70)

crowded = transit_df[
    transit_df["Occupancy (%)"] > 75
] if len(rows) else pd.DataFrame()

if len(crowded):

    print("Avoid these crowded transit hubs:\n")

    for _, row in crowded.iterrows():

        print(
            f"• {row['Transit Hub']} "
            f"({row['Occupancy (%)']}%)"
        )

else:

    print("✓ No heavily congested transit hubs nearby.")

low = transit_df[
    transit_df["Occupancy (%)"] <= 25
] if len(rows) else pd.DataFrame()

if len(low):

    best_station = low.iloc[0]

    print(
        f"\n✓ Recommended Transit Hub: "
        f"{best_station['Transit Hub']} "
        f"({best_station['Occupancy (%)']}%)"
    )

print(
    f"\n✓ Proceed to "
    f"{best['facility']} "
    "using the recommended evacuation route."
)

print("\n" + "="*70)


                 EVACUATION PLAN

🏥 RECOMMENDED FACILITY
----------------------------------------------------------------------


,Facility,Type,Distance (km),ETA (min),Score
0,General Williams Hospital,Hospital,2.26,5,7.22



🏥 ALTERNATIVE FACILITIES
----------------------------------------------------------------------


,Facility,Type,Distance (km),ETA (min),Score
0,Guru Harkrishan Polyclinic Bangla Sahib,Hospital,2.46,5,7.08
1,Press Club of India,Community Centre,1.71,3,7.00
2,Arvind Medicare,Hospital,2.63,5,6.96
3,Young Women's Christian Association,Community Centre,1.90,4,6.87
4,Dr. Ram Manohar Lohia Hospital,Hospital,2.86,6,6.80



🚇 NEARBY TRANSIT CONGESTION (PeopleSense)
----------------------------------------------------------------------


,Transit Hub,Distance (km),Occupancy (%),Status
0,DMRC,0.35,54.0,🟠 Busy
1,DMRC,0.44,50.0,🟡 Moderate
2,DMRC,1.15,76.0,🔴 Crowded
3,DMRC,1.80,34.0,🟡 Moderate
4,Rajiv Chowk F-Block,2.31,89.0,🔴 Crowded



⚠ TRAVEL ADVISORY
----------------------------------------------------------------------
Avoid these crowded transit hubs:

• DMRC (76.0%)
• Rajiv Chowk F-Block (89.0%)

✓ Proceed to General Williams Hospital using the recommended evacuation route.

